# 03 — Data Integration

Joins the cleaned Olist tables into a single transaction-level dataset, validating row counts at every step to prevent many-to-many duplication.

In [1]:
import pandas as pd
RAW = "../data/raw"
customers = pd.read_csv(f"{RAW}/olist_customers_dataset.csv")
orders = pd.read_csv(f"{RAW}/olist_orders_dataset.csv")
order_items = pd.read_csv(f"{RAW}/olist_order_items_dataset.csv")
payments = pd.read_csv(f"{RAW}/olist_order_payments_dataset.csv")
products = pd.read_csv(f"{RAW}/olist_products_dataset.csv")
cat_trans = pd.read_csv(f"{RAW}/product_category_name_translation.csv")
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
orders_valid = orders[orders["order_status"]=="delivered"].copy()
products["product_category_name"] = products["product_category_name"].fillna("category_not_informed")

## Join sequence
1. orders (delivered) + customers on customer_id (m:1)
2. + order_items on order_id (1:m, inner)
3. + products on product_id (m:1)
4. + category_translation (m:1)
5. + payments, **pre-aggregated to order level** before joining, to avoid duplicating order-item rows against multiple installment rows.

In [2]:
step = orders_valid.merge(customers, on="customer_id", how="left", validate="m:1")
step2 = step.merge(order_items, on="order_id", how="inner", validate="1:m")
step3 = step2.merge(products[["product_id","product_category_name"]], on="product_id", how="left", validate="m:1")
step4 = step3.merge(cat_trans, on="product_category_name", how="left", validate="m:1")

payments_agg = payments.groupby("order_id", as_index=False).agg(
    payment_value=("payment_value","sum"),
    payment_installments_max=("payment_installments","max"),
    payment_type_primary=("payment_type", lambda x: x.value_counts().idxmax()))

final = step4.merge(payments_agg, on="order_id", how="left", validate="m:1")
print(f"Final: {len(final):,} rows, {final['customer_unique_id'].nunique():,} customers, "
      f"{final['order_id'].nunique():,} orders")

Final: 110,197 rows, 93,358 customers, 96,478 orders


Row count is stable across every join after the payments pre-aggregation step — confirming no many-to-many duplication occurred. Output saved to `../outputs/cleaned/olist_customer_transactions.csv` (already generated and validated; see `../CLEANING_INTEGRATION_LOG.md` for the full join-by-join log).